In [1]:
import numpy as np
from torch.utils.data import Subset
from torchvision import datasets, transforms

# 1. Prepare your dataset (example: CIFAR-10)
tfm = transforms.ToTensor()
full_dataset = datasets.CIFAR10(root="./data", train=True, download=True, transform=tfm)
n = len(full_dataset)
print(n)

50000


In [2]:
# 2. Use NumPy RNG with a fixed seed to generate a consistent permutation
rng = np.random.default_rng(seed=42)   # <-- fixed for all experiments
indices = rng.permutation(n)
print(indices)
print(len(indices))

[  712 24774 45142 ...  5668 22113  9864]
50000


In [3]:
np.random.seed(42)
subset_indices = np.random.choice(n, size=n, replace=False)
print(subset_indices)

[33553  9427   199 ... 38158   860 15795]


In [4]:
train_idx = indices[:45000]
in_sample_idx = indices[:10000]
val_idx   = indices[45000:]

In [5]:
train_ds = Subset(full_dataset, train_idx)
val_ds   = Subset(full_dataset, val_idx)
in_sample_ds  = Subset(full_dataset, in_sample_idx)

In [2]:
import os
import glob
from pathlib import Path
import torch
import torch.nn as nn
from Model.ResNet_18 import ResNet18
from Model.VGG16 import ModifiedVGG16
import torch.backends.cudnn as cudnn 
import random
import numpy as np
from Dataset.CIFAR_10 import CIFAR10Dataset
from Dataset.CIFAR_100 import CIFAR100Dataset
from torch.utils.data import Subset, DataLoader

In [155]:
DATASET = ["CIFAR-100"]#"CIFAR-10"]  # , "MNIST", "CIFAR-100"
MODEL = ["ResNet-18"] #,
SEED = [46]

MODEL_DIR = Path("./saved_models/normal_new")
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

In [156]:
def load_last_checkpoint(model_dir: Path):
    """
    Load the most recent .pth checkpoint in a model directory.
    Returns the checkpoint path or None if not found.
    """
    ckpt_files = sorted(model_dir.glob("*.pth"), key=os.path.getmtime)
    if not ckpt_files:
        return None
    last_ckpt = ckpt_files[-1]  # the newest one
    #print(f"[📦] Loading last checkpoint: {last_ckpt.name}")
    return last_ckpt

In [157]:
for dataset in DATASET:
        for model in MODEL:
            for seed in SEED:
                # each run is saved in its own folder, e.g. ./saved_models/CIFAR-10_ResNet-18_45/
                folder_name = f"{dataset}_{model}_{seed}"
                model_folder = MODEL_DIR / folder_name

                if not model_folder.exists():
                    print(f"[⚠️] Missing model folder: {model_folder}")
                    continue

                # find last checkpoint
                ckpt_path = load_last_checkpoint(model_folder)
                if ckpt_path is None:
                    print(f"[⚠️] No .pth files found in {model_folder}")
                    continue
                
                num_classes = 10 if dataset == "CIFAR-10" else 100

In [158]:
ckpt_path

WindowsPath('saved_models/normal_new/CIFAR-100_ResNet-18_46/epoch_199.pth')

In [11]:
class vgg16_conv_block(nn.Module):
    def __init__(self, input_channels, out_channels, rate=0.3, drop=True):
        super().__init__()
        self.conv = nn.Conv2d(input_channels, out_channels, 3 ,1, 1)
        self.bn = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU(inplace=True)
        self.dropout = nn.Dropout(rate)
        self.drop =drop

    def forward(self, x):
        x = self.relu(self.bn(self.conv(x)))
        if self.drop:
            x = self.dropout(x)
        return(x)

def vgg16_layer(input_channels, out_channels, num, dropout=[0.3, 0.3]):
    """
    Creates a sequence of convolutional blocks + maxpool, as in VGG16.
    Args:
        input_channels: input channels to the first block
        out_channels: number of output channels for this layer
        num_blocks: how many conv blocks before pooling
        dropout: [drop_rate_first, drop_rate_later]
    """

    layers = []
    layers.append(vgg16_conv_block(input_channels, out_channels, dropout[0]))
    for i in range(1, num-1):
        layers.append(vgg16_conv_block(out_channels, out_channels, dropout[1]))
    if num>1:
        layers.append(vgg16_conv_block(out_channels, out_channels, drop=False))
    layers.append(nn.MaxPool2d(2,2))
    return(layers)


class ModifiedVGG16(nn.Module):
    """
        Modified VGG16 for CIFAR-10 or CIFAR-100.
        Args:
            num_classes (int): number of output classes (10 or 100 typically)
    """
    def __init__(self, num_classes=100):
        super(ModifiedVGG16, self).__init__()
        self.features = nn.Sequential(*vgg16_layer(3,64,2,[0.3,0.3]), *vgg16_layer(64,128,2),
                                      *vgg16_layer(128,256,3), *vgg16_layer(256,512,3),
                                      *vgg16_layer(512,512,3))
        self.classifier = nn.Sequential(nn.Dropout(0.2), 
                                        nn.Flatten(), 
                                        nn.Linear(512, 512, bias=True),
                                        nn.BatchNorm1d(512), 
                                        nn.ReLU(inplace=True), 
                                        nn.Linear(512,num_classes, bias=True))
         
    def forward(self, x):
        x = self.features(x)
        x = self.classifier[0](x)  
        x = self.classifier[1](x) 
        x = self.classifier[2](x)   
        x = self.classifier[3](x)  
        x = self.classifier[4](x)  
        x = self.classifier[5](x)     
        return x   

In [159]:
from Model.ResNet_18 import ResNet18

net = ResNet18(num_classes=num_classes).to(DEVICE)

In [160]:
state = torch.load(ckpt_path, map_location=DEVICE)

In [13]:
net = ModifiedVGG16(num_classes=num_classes).to(DEVICE)

In [161]:
net.load_state_dict(state)

<All keys matched successfully>

In [ ]:
for name, param in net.named_parameters():
    print(f"{name}: f{param}")

In [15]:
if hasattr(net, 'fc'):  # ResNet / DenseNet style
    head_attr = 'fc'
    head_module = net.fc
elif hasattr(net, 'classifier'):  # VGG / AlexNet style
    head_attr = 'classifier'
    head_module = net.classifier[-1]  # final Linear layer

print(head_attr)
print(head_module)

classifier
Linear(in_features=512, out_features=10, bias=True)


In [16]:
if isinstance(head_module, nn.Linear):
    current_out = head_module.out_features #10, 100
    current_in = head_module.in_features #512

print(current_out)
print(current_in)

10
512


In [17]:
for p in net.parameters():
    
    p.requires_grad = False

for p in head_module.parameters():
    p.requires_grad = True

In [ ]:
for name, param in net.named_parameters():
    print(f"{name}: requires_grad={param.requires_grad}")

In [ ]:
for name, module in net.named_modules():
        print(name, module)


In [36]:
new_head = nn.Linear(current_in, num_classes)

In [41]:
print("Weight of the linear layer:")
print(new_head.weight)

print("\nBias of the linear layer:")
print(new_head.bias)

Weight of the linear layer:
Parameter containing:
tensor([[ 0.0263, -0.0035, -0.0106,  ..., -0.0278,  0.0015, -0.0021],
        [-0.0258,  0.0311, -0.0235,  ...,  0.0201,  0.0075, -0.0034],
        [ 0.0425, -0.0347,  0.0372,  ...,  0.0372, -0.0284,  0.0346],
        ...,
        [-0.0066, -0.0081, -0.0039,  ...,  0.0268,  0.0202, -0.0188],
        [-0.0354, -0.0267, -0.0002,  ...,  0.0283, -0.0254,  0.0276],
        [ 0.0300, -0.0313, -0.0117,  ...,  0.0347,  0.0029,  0.0182]],
       requires_grad=True)

Bias of the linear layer:
Parameter containing:
tensor([-0.0358, -0.0207,  0.0051,  0.0033, -0.0366,  0.0221, -0.0004, -0.0170,
        -0.0271,  0.0007, -0.0162, -0.0027,  0.0021, -0.0277, -0.0235,  0.0291,
        -0.0415, -0.0322, -0.0190,  0.0092,  0.0065, -0.0268, -0.0361,  0.0287,
        -0.0432,  0.0420,  0.0010, -0.0024,  0.0272, -0.0422, -0.0332,  0.0147,
         0.0179,  0.0242,  0.0151, -0.0215,  0.0439,  0.0104,  0.0386, -0.0347,
         0.0404, -0.0154, -0.0355, -0.01

In [42]:
nn.init.xavier_uniform_(new_head.weight)
nn.init.zeros_(new_head.bias)


Parameter containing:
tensor([0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0.], requires_grad=True)

In [43]:
print("Weight of the linear layer:")
print(new_head.weight)

print("\nBias of the linear layer:")
print(new_head.bias)

Weight of the linear layer:
Parameter containing:
tensor([[-0.0729,  0.0195,  0.0941,  ...,  0.0234, -0.0053,  0.0482],
        [-0.0874,  0.0650,  0.0636,  ...,  0.0885, -0.0968,  0.0194],
        [-0.0813,  0.0004, -0.0041,  ..., -0.0968, -0.0523, -0.0412],
        ...,
        [-0.0746, -0.0568, -0.0353,  ..., -0.0669, -0.0636,  0.0473],
        [ 0.0128,  0.0660, -0.0460,  ..., -0.0085, -0.0723,  0.0130],
        [-0.0257,  0.0911,  0.0256,  ...,  0.0357,  0.0209, -0.0404]],
       requires_grad=True)

Bias of the linear layer:
Parameter containing:
tensor([0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0.],

## Model Pruning

In [31]:
parameters_to_prune = []
for name, module in net.named_modules():
    if isinstance(module, nn.Conv2d) or isinstance(module, nn.Linear):
        parameters_to_prune.append((module, 'weight'))

In [23]:
parameters_to_prune

[(Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1)), 'weight'),
 (Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1)), 'weight'),
 (Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1)),
  'weight'),
 (Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1)),
  'weight'),
 (Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1)),
  'weight'),
 (Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1)),
  'weight'),
 (Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1)),
  'weight'),
 (Conv2d(256, 512, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1)),
  'weight'),
 (Conv2d(512, 512, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1)),
  'weight'),
 (Conv2d(512, 512, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1)),
  'weight'),
 (Conv2d(512, 512, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1)),
  'weight'),
 (Conv2d(512, 512, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1)),
  'weight'),
 (

In [32]:
import torch.nn.utils.prune as prune

prune.global_unstructured(
        parameters_to_prune,
        pruning_method=prune.L1Unstructured,
        amount=0.2
    )

# Optionally remove the pruning reparameterization (make zeros permanent)
for module, _ in parameters_to_prune:
    prune.remove(module, 'weight')

In [ ]:
print(list(net.named_parameters()))

In [20]:
def check_pruned_weights(model):
    """
    Inspect the sparsity (percentage of zero weights) in each pruned layer.
    Also prints global sparsity across the entire model.
    """
    total_weights = 0
    total_zero = 0

    print("\n=== PRUNING CHECKER ===")
    for name, module in model.named_modules():

        # Only examine Conv2d and Linear layers (pruned layers)
        if isinstance(module, nn.Conv2d) or isinstance(module, nn.Linear):
            if hasattr(module, "weight"):
                w = module.weight.data.cpu().numpy()
                num_total = w.size
                num_zero = (w == 0).sum()

                layer_sparsity = num_zero / num_total * 100

                print(f"{name}.weight: zero = {num_zero}/{num_total}  "
                      f"({layer_sparsity:.2f}% sparsity)")

                total_weights += num_total
                total_zero += num_zero

    global_sparsity = total_zero / total_weights * 100
    print(f"\nGLOBAL SPARSITY: {global_sparsity:.2f}%\n")
    return global_sparsity


In [21]:
check_pruned_weights(net)


=== PRUNING CHECKER ===
features.0.conv.weight: zero = 0/1728  (0.00% sparsity)
features.1.conv.weight: zero = 0/36864  (0.00% sparsity)
features.3.conv.weight: zero = 0/73728  (0.00% sparsity)
features.4.conv.weight: zero = 0/147456  (0.00% sparsity)
features.6.conv.weight: zero = 0/294912  (0.00% sparsity)
features.7.conv.weight: zero = 0/589824  (0.00% sparsity)
features.8.conv.weight: zero = 0/589824  (0.00% sparsity)
features.10.conv.weight: zero = 0/1179648  (0.00% sparsity)
features.11.conv.weight: zero = 0/2359296  (0.00% sparsity)
features.12.conv.weight: zero = 0/2359296  (0.00% sparsity)
features.14.conv.weight: zero = 0/2359296  (0.00% sparsity)
features.15.conv.weight: zero = 0/2359296  (0.00% sparsity)
features.16.conv.weight: zero = 0/2359296  (0.00% sparsity)
classifier.2.weight: zero = 0/262144  (0.00% sparsity)
classifier.5.weight: zero = 0/5120  (0.00% sparsity)

GLOBAL SPARSITY: 0.00%



np.float64(0.0)

In [30]:
check_pruned_weights(net)


=== PRUNING CHECKER ===
features.0.conv.weight: zero = 0/1728  (0.00% sparsity)
features.1.conv.weight: zero = 0/36864  (0.00% sparsity)
features.3.conv.weight: zero = 0/73728  (0.00% sparsity)
features.4.conv.weight: zero = 0/147456  (0.00% sparsity)
features.6.conv.weight: zero = 0/294912  (0.00% sparsity)
features.7.conv.weight: zero = 0/589824  (0.00% sparsity)
features.8.conv.weight: zero = 0/589824  (0.00% sparsity)
features.10.conv.weight: zero = 0/1179648  (0.00% sparsity)
features.11.conv.weight: zero = 0/2359296  (0.00% sparsity)
features.12.conv.weight: zero = 0/2359296  (0.00% sparsity)
features.14.conv.weight: zero = 0/2359296  (0.00% sparsity)
features.15.conv.weight: zero = 0/2359296  (0.00% sparsity)
features.16.conv.weight: zero = 0/2359296  (0.00% sparsity)
classifier.2.weight: zero = 0/262144  (0.00% sparsity)
classifier.5.weight: zero = 0/51200  (0.00% sparsity)

GLOBAL SPARSITY: 0.00%



np.float64(0.0)

In [36]:
check_pruned_weights(net)


=== PRUNING CHECKER ===
conv1.weight: zero = 0/1728  (0.00% sparsity)
layer1.0.conv1.weight: zero = 0/36864  (0.00% sparsity)
layer1.0.conv2.weight: zero = 0/36864  (0.00% sparsity)
layer1.1.conv1.weight: zero = 0/36864  (0.00% sparsity)
layer1.1.conv2.weight: zero = 0/36864  (0.00% sparsity)
layer2.0.conv1.weight: zero = 0/73728  (0.00% sparsity)
layer2.0.conv2.weight: zero = 0/147456  (0.00% sparsity)
layer2.0.downsample.0.weight: zero = 0/8192  (0.00% sparsity)
layer2.1.conv1.weight: zero = 0/147456  (0.00% sparsity)
layer2.1.conv2.weight: zero = 0/147456  (0.00% sparsity)
layer3.0.conv1.weight: zero = 0/294912  (0.00% sparsity)
layer3.0.conv2.weight: zero = 0/589824  (0.00% sparsity)
layer3.0.downsample.0.weight: zero = 0/32768  (0.00% sparsity)
layer3.1.conv1.weight: zero = 0/589824  (0.00% sparsity)
layer3.1.conv2.weight: zero = 0/589824  (0.00% sparsity)
layer4.0.conv1.weight: zero = 0/1179648  (0.00% sparsity)
layer4.0.conv2.weight: zero = 1/2359296  (0.00% sparsity)
layer4.0.

np.float64(8.957080536335651e-06)

In [56]:
state = torch.load(ckpt_path, map_location=DEVICE)
net = ModifiedVGG16(num_classes=num_classes).to(DEVICE)
net.load_state_dict(state)

<All keys matched successfully>

In [ ]:
from Dataset.CIFAR_10 import CIFAR10Dataset

dataset_10 = CIFAR10Dataset()

train_set = dataset_10.train_set
test_set = dataset_10.test_set

in_eval_set = dataset_10.in_sample_set

def create_train_subset(train_set, length, seed):
    n = len(train_set)
    rng = np.random.default_rng(seed)
    perm = rng.permutation(n)

    idx_a, idx_b = perm[:length], perm[length:]

    ds_a = Subset(train_set, idx_a)
    ds_b = Subset(train_set, idx_b)

    # show sizes
    print(f"Created subsets: ds_a={len(ds_a)}, ds_b={len(ds_b)} (total={n})")

    return ds_a, ds_b

train_subset1, train_subset2 = create_train_subset(in_eval_set, 50, 42)

def sample_subset(dataset, sample_size, seed):
    """
    Randomly sample `sample_size` items from a given dataset.
    The selection is reproducible given the same seed.
    """
    n = len(dataset)
    rng = np.random.default_rng(seed)
    indices = rng.choice(n, size=sample_size, replace=False)
    print(len(indices))
    print(indices)
    return Subset(dataset, indices)

train_mask = sample_subset(train_subset1, 40000, seed=42)
print(len(train_mask))

trainloader = DataLoader(train_mask, batch_size=64, shuffle=False) # 100 samples

In [173]:
dataset_100 = CIFAR100Dataset()

train_set = dataset_100.train_set
test_set = dataset_100.test_set

in_eval_set = dataset_100.in_sample_set

def create_train_subset(train_set, length, seed):
    n = len(train_set)
    rng = np.random.default_rng(seed)
    perm = rng.permutation(n)

    idx_a, idx_b = perm[:length], perm[length:]

    ds_a = Subset(train_set, idx_a)
    ds_b = Subset(train_set, idx_b)

    # show sizes
    print(f"Created subsets: ds_a={len(ds_a)}, ds_b={len(ds_b)} (total={n})")

    return ds_a, ds_b

train_subset1, train_subset2 = create_train_subset(in_eval_set, 50, 42)

def sample_subset(dataset, sample_size, seed):
    """
    Randomly sample `sample_size` items from a given dataset.
    The selection is reproducible given the same seed.
    """
    n = len(dataset)
    rng = np.random.default_rng(seed)
    indices = rng.choice(n, size=sample_size, replace=False)
    print(len(indices))
    print(indices)
    return Subset(dataset, indices)

train_mask = sample_subset(train_subset1, 50, seed=42)
print(len(train_mask))

trainloader = DataLoader(train_mask, batch_size=128, shuffle=False) # 100 samples

Created subsets: ds_a=50, ds_b=49950 (total=50000)
50
[18 15  0 26 45 16 39 41 20 10  8 25 43 32 47 19 40 44 24 36 48  4  7 31
 46  3 12  5 42 49  2 33 29 11 14 34 13 28 35 27 30  6 23  1 22 21 17  9
 37 38]
50


In [177]:
num_classes=100
value_xt, value_ty = MI_cal_v2(label_matrix.numpy(), layer_T.numpy(), 50)
end_time = time.time()
elapsed_time = end_time - start_time

In [175]:
layer_T, label_matrix = None, None

import torch.nn.functional as F
import pandas as pd
# collect layer_T_array for MI analysis
def calculate_MI_input(layer_T_array, label_matrix, outputs, targets, num_classes):
    outputs_cpu = outputs.detach().cpu()
    targets_cpu = targets.detach().cpu()


    if layer_T_array is None:
        layer_T_array = outputs.detach().cpu()
    else:
        layer_T_array = torch.cat((layer_T_array, outputs_cpu), dim=0)

    # accumulate one-hot labels
    batch_onehot = F.one_hot(targets_cpu, num_classes=num_classes).float().cpu()

    if label_matrix is None:
        label_matrix = batch_onehot
    else:
        label_matrix = torch.cat((label_matrix, batch_onehot), dim=0)
    
    return layer_T_array, label_matrix

import time

start_time = time.time()

net.eval()
with torch.no_grad():
    for batch_idx, (inputs, targets) in enumerate(trainloader):
        inputs, targets = inputs.to(DEVICE), targets.to(DEVICE)
        outputs = net(inputs)

        layer_T, label_matrix = calculate_MI_input(layer_T, label_matrix
                                                        ,outputs, targets, num_classes)



In [178]:
value_xt

np.float64(5.348758439731457)

In [179]:
value_ty

np.float64(5.173660689688187)

In [163]:
layer_T = layer_T.numpy()
label_matrix = label_matrix.numpy()

In [ ]:
print(value_xt)
print(value_ty)
print(f"Time taken: {elapsed_time:.4f} seconds")

In [108]:
layer_T.shape

(50, 100)

In [164]:
layer_T = np.exp(layer_T - np.max(layer_T,axis=1,keepdims=True))
layer_T /= np.sum(layer_T,axis=1,keepdims=True)

print(layer_T)

[[3.22211321e-07 4.55029095e-07 4.79839599e-08 ... 6.42255600e-08
  6.13371967e-08 6.52614887e-08]
 [2.02309981e-04 7.86913151e-05 7.13436675e-05 ... 1.91683401e-04
  6.96274801e-05 3.26921028e-04]
 [1.21268195e-05 1.16390165e-05 1.30197486e-05 ... 1.61803691e-04
  6.00121302e-06 3.12570373e-05]
 ...
 [2.16460398e-06 1.27992450e-06 8.43517341e-07 ... 1.76751917e-06
  1.45694048e-06 2.37610357e-06]
 [2.69993434e-05 7.12426991e-05 3.46057095e-05 ... 1.66623344e-04
  1.49076732e-04 3.04586902e-05]
 [3.47152331e-06 9.42016050e-06 1.87211699e-05 ... 1.16544819e-04
  3.14003228e-05 1.58338898e-05]]


In [165]:
labels = np.arange(50)
bins = np.arange(51)
bins = bins/float(50)

In [99]:
print(labels)
print(bins)

[ 0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19 20 21 22 23
 24 25 26 27 28 29 30 31 32 33 34 35 36 37 38 39 40 41 42 43 44 45 46 47
 48 49]
[0.   0.02 0.04 0.06 0.08 0.1  0.12 0.14 0.16 0.18 0.2  0.22 0.24 0.26
 0.28 0.3  0.32 0.34 0.36 0.38 0.4  0.42 0.44 0.46 0.48 0.5  0.52 0.54
 0.56 0.58 0.6  0.62 0.64 0.66 0.68 0.7  0.72 0.74 0.76 0.78 0.8  0.82
 0.84 0.86 0.88 0.9  0.92 0.94 0.96 0.98 1.  ]


In [166]:
layer_T = Discretize_v2(layer_T)

In [ ]:
print(layer_T[22:32])

In [167]:
XT_matrix = np.zeros((50,50))

Non_repeat, mark_list = [], []

for i in range(50):
    pre_mark_size = len(mark_list)
    if i==0:
        Non_repeat.append(i)
        mark_list.append(i)
        XT_matrix[i,i]=1
    else:
        for j in range(len(Non_repeat)):
            if (layer_T[i] ==layer_T[ Non_repeat[j] ]).all():
                mark_list.append(Non_repeat[j])
                XT_matrix[i,Non_repeat[j]]=1
                break
    if pre_mark_size == len(mark_list):
        Non_repeat.append(Non_repeat[-1]+1)
        mark_list.append(Non_repeat[-1])
        XT_matrix[i,Non_repeat[-1]]=1


In [136]:
XT_matrix

array([[1., 0., 0., ..., 0., 0., 0.],
       [0., 1., 0., ..., 0., 0., 0.],
       [0., 0., 1., ..., 0., 0., 0.],
       ...,
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.]], shape=(50, 50))

In [137]:
print(Non_repeat)

[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38]


In [172]:
print(mark_list)
print(len(mark_list))

[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 10, 12, 4, 13, 14, 15, 11, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 28, 32, 33, 34, 35, 36, 37, 28, 38, 39, 40, 2, 41, 41, 42]
50


In [168]:
XT_matrix = np.delete(XT_matrix,range(len(Non_repeat),50),axis=1)	
XT_matrix = XT_matrix/50

In [141]:
XT_matrix

array([[0.02, 0.  , 0.  , ..., 0.  , 0.  , 0.  ],
       [0.  , 0.02, 0.  , ..., 0.  , 0.  , 0.  ],
       [0.  , 0.  , 0.02, ..., 0.  , 0.  , 0.  ],
       ...,
       [0.  , 0.  , 0.  , ..., 0.  , 0.02, 0.  ],
       [0.  , 0.  , 0.  , ..., 0.  , 0.  , 0.02],
       [0.  , 0.  , 0.  , ..., 0.  , 0.  , 0.  ]], shape=(50, 39))

In [169]:
P_X_for_layer_T = np.sum(XT_matrix,axis=1)
P_layer_T_for_X= np.sum(XT_matrix,axis=0)

In [171]:
P_X_for_layer_T

array([0.02, 0.02, 0.02, 0.02, 0.02, 0.02, 0.02, 0.02, 0.02, 0.02, 0.02,
       0.02, 0.02, 0.02, 0.02, 0.02, 0.02, 0.02, 0.02, 0.02, 0.02, 0.02,
       0.02, 0.02, 0.02, 0.02, 0.02, 0.02, 0.02, 0.02, 0.02, 0.02, 0.02,
       0.02, 0.02, 0.02, 0.02, 0.02, 0.02, 0.02, 0.02, 0.02, 0.02, 0.02,
       0.02, 0.02, 0.02, 0.02, 0.02, 0.02])

In [170]:
P_layer_T_for_X

array([0.02, 0.02, 0.04, 0.02, 0.04, 0.02, 0.02, 0.02, 0.02, 0.02, 0.04,
       0.04, 0.02, 0.02, 0.02, 0.02, 0.02, 0.02, 0.02, 0.02, 0.02, 0.02,
       0.02, 0.02, 0.02, 0.02, 0.02, 0.02, 0.06, 0.02, 0.02, 0.02, 0.02,
       0.02, 0.02, 0.02, 0.02, 0.02, 0.02, 0.02, 0.02, 0.04, 0.02])

In [146]:
MI_XT=0
for i in range(XT_matrix.shape[0]):
    for j in range(XT_matrix.shape[1]):
        if XT_matrix[i,j]==0:
            pass
        else:
            MI_XT+=XT_matrix[i,j]*np.log2(XT_matrix[i,j]/
                                          (P_X_for_layer_T[i]*P_layer_T_for_X[j]))

In [147]:
MI_XT

np.float64(5.188758439731455)

In [150]:

MI_TY = 0
TY_matrix = np.zeros((len(Non_repeat),100))
mark_list = np.array(mark_list)
for i in range(len(Non_repeat)):
    TY_matrix[i,:] = np.sum(label_matrix[  np.where(mark_list==i)[0]  , : ] ,axis=0 )
TY_matrix = TY_matrix/50
P_layer_T_for_Y = np.sum(TY_matrix,axis=1)
P_Y_for_layer_T = np.sum(TY_matrix,axis=0)
for i in range(TY_matrix.shape[0]):
    for j in range(TY_matrix.shape[1]):
        if TY_matrix[i,j]==0:
            pass
        else:
            MI_TY+=TY_matrix[i,j]*np.log2(TY_matrix[i,j]/(P_layer_T_for_Y[i]*P_Y_for_layer_T[j]))

In [151]:
MI_TY

np.float64(5.038562939644918)

In [51]:
XT_matrix = XT_matrix/100
XT_matrix

array([[0.01, 0.  , 0.  , ..., 0.  , 0.  , 0.  ],
       [0.  , 0.01, 0.  , ..., 0.  , 0.  , 0.  ],
       [0.01, 0.  , 0.  , ..., 0.  , 0.  , 0.  ],
       ...,
       [0.01, 0.  , 0.  , ..., 0.  , 0.  , 0.  ],
       [0.01, 0.  , 0.  , ..., 0.  , 0.  , 0.  ],
       [0.  , 0.  , 0.  , ..., 0.  , 0.  , 0.  ]], shape=(100, 21))

In [52]:
P_X_for_layer_T = np.sum(XT_matrix,axis=1)
P_layer_T_for_X = np.sum(XT_matrix,axis=0)

print(P_X_for_layer_T)
print(P_layer_T_for_X)

[0.01 0.01 0.01 0.01 0.01 0.01 0.01 0.01 0.01 0.01 0.01 0.01 0.01 0.01
 0.01 0.01 0.01 0.01 0.01 0.01 0.01 0.01 0.01 0.01 0.01 0.01 0.01 0.01
 0.01 0.01 0.01 0.01 0.01 0.01 0.01 0.01 0.01 0.01 0.01 0.01 0.01 0.01
 0.01 0.01 0.01 0.01 0.01 0.01 0.01 0.01 0.01 0.01 0.01 0.01 0.01 0.01
 0.01 0.01 0.01 0.01 0.01 0.01 0.01 0.01 0.01 0.01 0.01 0.01 0.01 0.01
 0.01 0.01 0.01 0.01 0.01 0.01 0.01 0.01 0.01 0.01 0.01 0.01 0.01 0.01
 0.01 0.01 0.01 0.01 0.01 0.01 0.01 0.01 0.01 0.01 0.01 0.01 0.01 0.01
 0.01 0.01]
[0.13 0.08 0.01 0.11 0.1  0.14 0.01 0.01 0.1  0.01 0.01 0.01 0.07 0.05
 0.01 0.03 0.08 0.01 0.01 0.01 0.01]


In [60]:
NUM_INTERVALS=50
#NUM_LABEL=10
mask=10000

def MI_cal_v2(label_matrix, layer_T, NUM_TEST_MASK):
    NUM_LABEL = label_matrix.shape[1]

    MI_XT=0
    MI_TY=0
    # This part is to transfrom logits to probabilities through softmax.
    layer_T = np.exp(layer_T - np.max(layer_T,axis=1,keepdims=True)) #numerical stabilization: prevent potential overflow issues for exponentiation.
    layer_T /= np.sum( layer_T,axis=1,keepdims=True)
    
    layer_T = Discretize_v2(layer_T)
    XT_matrix = np.zeros((NUM_TEST_MASK,NUM_TEST_MASK)) # 1000*1000
    Non_repeat=[]
    mark_list=[]
    for i in range(NUM_TEST_MASK):
        pre_mark_size = len(mark_list)
        if i==0:
            Non_repeat.append(i)
            mark_list.append(i)
            XT_matrix[i,i]=1
        else:
            for j in range(len(Non_repeat)):
                if (layer_T[i] ==layer_T[ Non_repeat[j] ]).all():
                    mark_list.append(Non_repeat[j])
                    XT_matrix[i,Non_repeat[j]]=1
                    break
        if pre_mark_size == len(mark_list):
            Non_repeat.append(Non_repeat[-1]+1)
            mark_list.append(Non_repeat[-1])
            XT_matrix[i,Non_repeat[-1]]=1
    
    XT_matrix = np.delete(XT_matrix,range(len(Non_repeat),NUM_TEST_MASK),axis=1)				
    XT_matrix = XT_matrix/NUM_TEST_MASK
    P_X_for_layer_T = np.sum(XT_matrix,axis=1)
    P_layer_T_for_X= np.sum(XT_matrix,axis=0)
    for i in range(XT_matrix.shape[0]):
        for j in range(XT_matrix.shape[1]):
            if XT_matrix[i,j]==0:
                pass
            else:
                MI_XT+=XT_matrix[i,j]*np.log2(XT_matrix[i,j]/
                                              (P_X_for_layer_T[i]*P_layer_T_for_X[j]))

    TY_matrix = np.zeros((len(Non_repeat),NUM_LABEL))
    mark_list = np.array(mark_list)
    for i in range(len(Non_repeat)):
        TY_matrix[i,:] = np.sum(label_matrix[  np.where(mark_list==i)[0]  , : ] ,axis=0 )
    TY_matrix = TY_matrix/NUM_TEST_MASK
    P_layer_T_for_Y = np.sum(TY_matrix,axis=1)
    P_Y_for_layer_T = np.sum(TY_matrix,axis=0)
    for i in range(TY_matrix.shape[0]):
        for j in range(TY_matrix.shape[1]):
            if TY_matrix[i,j]==0:
                pass
            else:
                MI_TY+=TY_matrix[i,j]*np.log2(TY_matrix[i,j]/(P_layer_T_for_Y[i]*P_Y_for_layer_T[j]))
    
    return MI_XT,MI_TY



#将连续数据转换为离散数据
def Discretize_v2(layer_T):	
    labels = np.arange(NUM_INTERVALS) # [0, 1, 2, 3 ..., 49]
    bins = np.arange(NUM_INTERVALS+1) # [0, 1, 2, 3 ..., 49, 50]
    bins = bins/float(NUM_INTERVALS)  # [0, 0.02, 0.04, ..., 1.0]
    
    for i in range(layer_T.shape[1]):
        temp = pd.cut(layer_T[:,i],bins,labels=labels)
        layer_T[:,i] = np.array(temp)
    return layer_T

In [152]:
a = [18,  0, 17, 27, 25, 10, 36, 37, 31, 38,  3, 24, 22, 39, 11, 35, 26, 17,
         2, 30, 16, 32, 33, 21, 33, 23, 20, 15, 28, 31, 14, 36, 34, 12,  4, 17,
         0, 25, 29, 19, 31, 13,  8, 30,  9,  1,  2,  5,  7,  6]
b = [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 22, 24, 25, 26, 27, 8, 28, 29, 30, 31, 32, 17, 1, 4, 33, 34, 8, 10, 14, 19, 35, 36, 18, 37, 38, 34]

a.sort()
b.sort()
print(a)
print(b)

[0, 1, 1, 2, 3, 4, 4, 5, 6, 7, 7, 8, 9, 9, 10, 11, 12, 13, 14, 15, 15, 16, 17, 18, 18, 19, 19, 20, 21, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 33, 34, 35, 36, 37, 37, 37, 38]
[0, 1, 1, 2, 3, 4, 4, 5, 6, 7, 8, 8, 8, 9, 10, 10, 11, 12, 13, 14, 14, 15, 16, 17, 17, 18, 18, 19, 19, 20, 21, 22, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 34, 35, 36, 37, 38]


In [ ]:
def MI_formula_cal(matrix, p1, p2): #p1, p2 represents marginals
    mask = matrix > 0

    denom = p1[:, None] * p2[None, :]
    
    ratio = matrix / denom
    
    log_ratio = torch.log2(ratio)
    
    MI = (matrix * log_ratio)[mask].sum()

    return MI
    
def MI_cal_gpu(layer_T, label_matrix, num_intervals=50):
    start_time = time.time()
    device = layer_T.device
    layer_T = torch.softmax(layer_T, dim=1)

    bins = torch.linspace(0, 1, num_intervals + 1, device=device, dtype=torch.float64)
    layer_T_discrete = torch.bucketize(layer_T, bins, right=True) - 1
    layer_T_discrete = layer_T_discrete.clamp(0, num_intervals - 1)

    layer_T = layer_T_discrete.contiguous()
    N, C = layer_T.shape

    XT_matrix = torch.zeros((N, N), device=device, dtype=torch.float64)
    non_repeat, mark_list = [], []

    eq_matrix = (layer_T.unsqueeze(1) == layer_T.unsqueeze(0)).all(dim=-1)
    
    for i in range(N):
        pre_mark_size = len(mark_list)
    
        if i == 0:
            # First row always unique
            non_repeat.append(i)
            mark_list.append(i)
            XT_matrix[i, i] = 1
            continue
    
        # Compare row i to all existing non_repeat rows
        nonrep_indices = torch.tensor(non_repeat, device=device, dtype=torch.long)
        matches = eq_matrix[i, nonrep_indices]  # vector of True/False
    
        if matches.any():
            j = matches.nonzero()[0].item()  # First matched index in non_repeat
            match_idx = non_repeat[j]
            mark_list.append(match_idx)
            XT_matrix[i, match_idx] = 1
        else:
            # No match found → new unique pattern
            new_idx = non_repeat[-1] + 1
            non_repeat.append(new_idx)
            mark_list.append(new_idx)
            XT_matrix[i, new_idx] = 1

    N_unique = len(non_repeat)
    XT_matrix = XT_matrix[:, :N_unique]
    XT_matrix = XT_matrix / N

    P_X_for_layer_T = XT_matrix.sum(dim=1)
    P_layer_T_for_X = XT_matrix.sum(dim=0)

    I_X_T = MI_formula_cal(XT_matrix, P_X_for_layer_T, P_layer_T_for_X)

    mark_t = torch.tensor(mark_list, device=device, dtype=torch.long)
    K = len(non_repeat)
    
    TY_matrix = torch.zeros((K, label_matrix.shape[1]), 
                            device=device, dtype=torch.float64)
    
    TY_matrix.index_add_(0, mark_t, label_matrix)
    
    TY_matrix = TY_matrix / N

    P_layer_T_for_Y = TY_matrix.sum(dim=1)
    P_Y_for_layer_T = TY_matrix.sum(dim=0)

    I_T_Y = MI_formula_cal(TY_matrix, P_layer_T_for_Y, P_Y_for_layer_T)

    end_time = time.time()
    elapsed_time = end_time - start_time
    print(elapsed_time)

    return I_X_T.item(), I_T_Y.item()

    